In [1]:
import os
import warnings

import duckdb
import pandas as pd

warnings.filterwarnings("ignore")

In [2]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
# path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [3]:
sensor_loc = pd.read_csv(path + r"5. Source & Refrence Files\sensor_loc.csv",
                         dtype={"station_id": 'str'})
state_map = pd.read_csv(path + r"5. Source & Refrence Files\State_mapping.csv")


In [26]:
# folder = path + r"5. Source & Refrence Files\2024_traffic_data"
# out_dir = os.path.join(path, r"5. Source & Refrence Files\2024_traffic_parquet")
# os.makedirs(out_dir, exist_ok=True)
#
# # build mapping dict ONCE from your state_map df
# state_dict = state_map.set_index("state_code")["State"].to_dict()
#
# id_var_col = [
#     'record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
#     'travel_lane', 'year_record', 'month_record', 'day_record',
#     'day_of_week', 'restrictions'
# ]
#
# part = 0
#
# for filename in os.listdir(folder):
#     file_path = os.path.join(folder, filename)
#
#     if not filename.lower().endswith(".zip"):
#         continue
#
#     print(f"Opening ZIP: {filename}")
#
#     with zipfile.ZipFile(file_path, 'r') as z:
#         for inner_name in z.namelist():
#             if inner_name.endswith("/"):
#                 continue
#
#             print(f"  Processing inside ZIP: {inner_name}")
#
#             with z.open(inner_name) as f:
#                 # Read CSV from inside the ZIP
#                 df = pd.read_csv(
#                     f,
#                     delimiter="|",
#                     low_memory=False  # avoids dtype warning at cost of some RAM, OK for chunk
#                     # You can also pass dtype={} here if you know them
#                 )
#
#                 # Melt wide hours columns into long format
#                 df = pd.melt(
#                     df,
#                     id_vars=id_var_col,
#                     var_name="hours",
#                     value_name="traffic"
#                 )
#
#                 # Add state_name via map instead of big merge later
#                 df["State"] = df["state_code"].map(state_dict)
#                 df["station_id"] = df["station_id"].astype(str)
#
#                 # Save this chunk to Parquet and drop from memory
#                 out_path = os.path.join(out_dir, f"traffic_part_{part}.parquet")
#                 df.to_parquet(out_path, index=False)
#                 # print(f"    → wrote {out_path}")
#
#                 del df
#                 part += 1
#
# print(f"Done. Wrote {part} parquet files to {out_dir}")


Opening ZIP: apr_2024_ccs_data.zip
  Processing inside ZIP: AK_APR_2024 (TMAS).VOL
  Processing inside ZIP: AL_APR_2024 (TMAS).VOL
  Processing inside ZIP: AR_APR_2024 (TMAS).VOL
  Processing inside ZIP: AZ_APR_2024 (TMAS).VOL
  Processing inside ZIP: CA_APR_2024 (TMAS).VOL
  Processing inside ZIP: CO_APR_2024 (TMAS).VOL
  Processing inside ZIP: CT_APR_2024 (TMAS).VOL
  Processing inside ZIP: DC_APR_2024 (TMAS).VOL
  Processing inside ZIP: DE_APR_2024 (TMAS).VOL
  Processing inside ZIP: FL_APR_2024 (TMAS).VOL
  Processing inside ZIP: GA_APR_2024 (TMAS).VOL
  Processing inside ZIP: HI_APR_2024 (TMAS).VOL
  Processing inside ZIP: IA_APR_2024 (TMAS).VOL
  Processing inside ZIP: ID_APR_2024 (TMAS).VOL
  Processing inside ZIP: IL_APR_2024 (TMAS).VOL
  Processing inside ZIP: IN_APR_2024 (TMAS).VOL
  Processing inside ZIP: KS_APR_2024 (TMAS).VOL
  Processing inside ZIP: KY_APR_2024 (TMAS).VOL
  Processing inside ZIP: LA_APR_2024 (TMAS).VOL
  Processing inside ZIP: MA_APR_2024 (TMAS).VOL
  Pro

In [4]:
traffic_df = pd.DataFrame()

In [5]:
out_dir = os.path.join(path, r"5. Source & Refrence Files\2024_traffic_parquet")
traffic_parquet_glob = f"{out_dir}/traffic_part_*.parquet"

In [6]:
con = duckdb.connect()
con.register("sensor_loc", sensor_loc)

In [7]:
out_dir = os.path.join(path, r"5. Source & Refrence Files\2024_traffic_parquet")
traffic_parquet_glob = f"{out_dir}/traffic_part_*.parquet"

In [30]:
sensor_loc

,Latitude,Longitude,Functional Class,State,Station Id
0,36.719788,-104.535744,4R,NM,WHIT01
1,40.485651,-74.426438,3U,NJ,W41024
2,40.892000,-74.223000,3U,NJ,W21035
3,39.771974,-74.901328,3U,NJ,W07140
4,39.944480,-74.095954,3U,NJ,W06108
...,...,...,...,...,...
8055,34.710839,-76.736897,4U,NC,0A1501
8056,35.551341,-82.731234,1U,NC,0A1003
8057,33.937646,-78.555950,7U,NC,0A0901
8058,35.946988,-82.004388,3R,NC,0A0501


In [29]:
con.execute(f"""
    CREATE OR REPLACE TABLE traffic_matched AS
    SELECT
        t.*,                           -- all columns from traffic
        s.*
    FROM read_parquet('{traffic_parquet_glob}') AS t
    LEFT JOIN sensor_loc AS s
      ON t.station_id = s."Station Id"
     AND t.State      = s.State
    WHERE s."Latitude" IS NOT NULL
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
con.execute(f"""
    CREATE OR REPLACE TABLE traffic_unmatched  AS
    SELECT
        t.*,                           -- all columns from traffic
        s.*
    FROM read_parquet('{traffic_parquet_glob}') AS t
    LEFT JOIN sensor_loc AS s
      ON t.station_id = s."Station Id"
     AND t.State      = s.State
    WHERE s."Latitude" IS NULL
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
213

In [14]:
con.execute("""
            UPDATE traffic_unmatched
            SET station_id = ltrim(station_id, '0')
            """)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
con.execute("""select *
               from sensor_loc limit 5""").df()

,Latitude,Longitude,Functional Class,State,Station Id
0,36.719788,-104.535744,4R,NM,WHIT01
1,40.485651,-74.426438,3U,NJ,W41024
2,40.892000,-74.223000,3U,NJ,W21035
3,39.771974,-74.901328,3U,NJ,W07140
4,39.944480,-74.095954,3U,NJ,W06108


In [26]:
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN Latitude')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN Longitude')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN "Functional Class"')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN State_1')
con.execute('ALTER TABLE traffic_unmatched DROP COLUMN "Station Id"')


In [27]:
con.execute("""select *
               from traffic_unmatched limit 5""").df()

,record_type,state_code,f_system,station_id,travel_dir,travel_lane,year_record,month_record,day_record,day_of_week,restrictions,hours,traffic,State_1
0,V,15,3U,2,1,1,2024,4,1,2,NaN,hour_00,22,None
1,V,15,3U,2,1,1,2024,4,2,3,NaN,hour_00,27,None
2,V,15,3U,2,1,1,2024,4,3,4,NaN,hour_00,25,None
3,V,15,3U,2,1,1,2024,4,4,5,NaN,hour_00,20,None
4,V,15,3U,2,1,1,2024,4,5,6,NaN,hour_00,34,None


In [31]:
con.execute("""
            -- INSERT INTO traffic_matched
            SELECT t.*,
                   s.*
            FROM traffic_unmatched t
                     LEFT JOIN sensor_loc s
                               ON t.station_id = s."Station Id"
                                   AND t.State = s.State
            WHERE s."Latitude" IS NOT NULL limit 5
            """).df()


BinderException: Binder Error: Table "t" does not have a column named "State"

Candidate bindings: : "State_1", "state_code", "station_id", "travel_dir", "travel_lane"

LINE 9:  AND t.State      = s.State
             ^

In [8]:
combined_traffic_df_merge = pd.merge(combined_traffic_df, sensor_loc, left_on=["station_id", "State"],
                                     right_on=["Station Id", "State"], how="left")
traffic_df = pd.concat([traffic_df, combined_traffic_df_merge[~combined_traffic_df_merge["Latitude"].isnull()]],
                       ignore_index=True)
combined_traffic_df = combined_traffic_df_merge[combined_traffic_df_merge["Latitude"].isnull()][columns].copy()
print(f"Unmatched records are: {len(combined_traffic_df)}")

Unmatched records are: 809019


In [9]:
combined_traffic_df["station_id"] = combined_traffic_df["station_id"].str.lstrip("0")

In [10]:
combined_traffic_df_merge = pd.merge(combined_traffic_df, sensor_loc, left_on=["station_id", "State"],
                                     right_on=["Station Id", "State"], how="left")
traffic_df = pd.concat([traffic_df, combined_traffic_df_merge[~combined_traffic_df_merge["Latitude"].isnull()]],
                       ignore_index=True)
combined_traffic_df = combined_traffic_df_merge[combined_traffic_df_merge["Latitude"].isnull()][columns].copy()
print(f"Unmatched records are: {len(combined_traffic_df)}")

Unmatched records are: 668


In [11]:
combined_traffic_df["station_id"].unique()

array(['E131'], dtype=object)

In [12]:
traffic_df.columns

Index(['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
       'travel_lane', 'year_record', 'month_record', 'day_record',
       'day_of_week', 'hour_00', 'hour_01', 'hour_02', 'hour_03', 'hour_04',
       'hour_05', 'hour_06', 'hour_07', 'hour_08', 'hour_09', 'hour_10',
       'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16',
       'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22',
       'hour_23', 'restrictions', 'State', 'Unnamed: 2', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Latitude', 'Longitude', 'Functional Class',
       'Station Id'],
      dtype='object')

In [18]:
clean_col = ['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
             'travel_lane', 'year_record', 'month_record', 'day_record',
             'day_of_week', 'hour_00', 'hour_01', 'hour_02', 'hour_03', 'hour_04',
             'hour_05', 'hour_06', 'hour_07', 'hour_08', 'hour_09', 'hour_10',
             'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16',
             'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22',
             'hour_23', 'restrictions', 'State', 'Latitude', 'Longitude', 'Functional Class',
             'Station Id']

In [19]:
traffic_df = traffic_df[clean_col].copy()

In [20]:
id_var_col = ['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
              'travel_lane', 'year_record', 'month_record', 'day_record',
              'day_of_week', 'restrictions', 'State', 'Latitude', 'Longitude', 'Functional Class',
              'Station Id']

In [21]:
traffic_melt_df = pd.melt(traffic_df, id_vars=id_var_col, var_name='hours', value_name='traffic')

MemoryError: Unable to allocate 16.8 GiB for an array with shape (12, 187530144) and data type object